# 六模型 RMSE/MAE 对比

该 notebook 对 6 个模型做统一 eval-only 对比，并输出两套数据集上的指标与最佳模型。

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from ase.io import read
from sklearn.metrics import mean_squared_error, mean_absolute_error

LOREM_ROOT = Path('/data/home/public/qiuqizhi/LOREM')
LOREM_CODE = LOREM_ROOT / 'lorem'
if str(LOREM_CODE) not in sys.path:
    sys.path.insert(0, str(LOREM_CODE))

from calculator import Calculator


In [2]:
MODEL_DIRS = {
    'SOG-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2'),
    # 'SOG-mp2-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run2'),
    # 'SOG-mp2-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-run3'),
    'CU-mp2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2'),
    # 'CU-mp2-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2-run2'),
    # 'SOG-mp1-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run2'),
    'SOG-mp1': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run3'),
    # 'SOG-mp1-run4': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run4'),
    # 'CU-mp1': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1'),
    'CU-mp1': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1-run2'),
    'SOG-mp2-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent'),
    # 'SOG-mp2-ldependent-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run2'),
    # 'SOG-mp2-ldependent-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run3'),
    # 'SOG-mp2-ldependent-run4': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent-run4'),
    'SOG-mp1-ldependent': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent'),
    # 'SOG-mp1-ldependent-run2': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent-run2'),
    # 'SOG-mp1-ldependent-run3': Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent-run3'),
}

CKPTS = {k: v / 'run/checkpoints/R2_E+F' for k, v in MODEL_DIRS.items()}
TEST_XYZ = LOREM_ROOT / 'datasets' / 'cumulene_test.xyz'
PROFILE_XYZ = LOREM_ROOT / 'datasets' / 'cumulene_profile.xyz'

for name, ckpt in CKPTS.items():
    for p in [
        ckpt / 'model/model.msgpack',
        ckpt / 'model/model.yaml',
        ckpt / 'model/baseline.yaml',
    ]:
        if not p.exists():
            raise FileNotFoundError(f'[{name}] Missing: {p}')

for p in [TEST_XYZ, PROFILE_XYZ]:
    if not p.exists():
        raise FileNotFoundError(f'Missing dataset: {p}')

print('六模型 checkpoint 已就绪:')
for name, ckpt in CKPTS.items():
    print(f'- {name}: {ckpt}')
print('Test dataset   :', TEST_XYZ)
print('Profile dataset:', PROFILE_XYZ)


六模型 checkpoint 已就绪:
- SOG-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2/run/checkpoints/R2_E+F
- CU-mp2: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp2/run/checkpoints/R2_E+F
- SOG-mp1: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-run3/run/checkpoints/R2_E+F
- CU-mp1: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1-run2/run/checkpoints/R2_E+F
- SOG-mp2-ldependent: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp2-ldependent/run/checkpoints/R2_E+F
- SOG-mp1-ldependent: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1-ldependent/run/checkpoints/R2_E+F
Test dataset   : /data/home/public/qiuqizhi/LOREM/datasets/cumulene_test.xyz
Profile dataset: /data/home/public/qiuqizhi/LOREM/datasets/cumulene_profile.xyz


In [4]:
def eval_checkpoint_on_xyz(ckpt_dir: Path, xyz_path: Path, add_offset: bool = True):
    calc = Calculator.from_checkpoint(ckpt_dir, add_offset=add_offset)
    systems = read(xyz_path, index=':')

    e_ref = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_ref = np.concatenate([s.get_forces() for s in systems], axis=0)

    for s in systems:
        s.calc = calc

    e_pred = np.array([s.get_potential_energy() / len(s) for s in systems], dtype=float)
    f_pred = np.concatenate([s.get_forces() for s in systems], axis=0)

    return {
        'energy_rmse_meV_per_atom': float(np.sqrt(mean_squared_error(e_ref, e_pred)) * 1000.0),
        'energy_mae_meV_per_atom': float(mean_absolute_error(e_ref, e_pred) * 1000.0),
        'force_rmse_meV_per_A': float(np.sqrt(mean_squared_error(f_ref, f_pred)) * 1000.0),
        'force_mae_meV_per_A': float(mean_absolute_error(f_ref, f_pred) * 1000.0),
    }


def eval_suite(model_ckpts: dict, xyz_path: Path, add_offset: bool = True):
    rows = []
    for name, ckpt in model_ckpts.items():
        metrics = eval_checkpoint_on_xyz(ckpt, xyz_path, add_offset=add_offset)
        rows.append({'model': name, **metrics})
    return pd.DataFrame(rows)


metrics_cols = [
    'energy_mae_meV_per_atom',
    'energy_rmse_meV_per_atom',
    'force_mae_meV_per_A',
    'force_rmse_meV_per_A',
]

# 在 test 数据集上比较
test_df = eval_suite(CKPTS, TEST_XYZ, add_offset=True)
print('=== model eval on cumulene_test.xyz ===')
display(test_df.sort_values('energy_rmse_meV_per_atom'))

# 在 profile 数据集上比较
profile_df = eval_suite(CKPTS, PROFILE_XYZ, add_offset=True)
print('=== model eval on cumulene_profile.xyz ===')
display(profile_df.sort_values('energy_rmse_meV_per_atom'))

best_test = {m: test_df.loc[test_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}
best_profile = {m: profile_df.loc[profile_df[m].idxmin(), ['model', m]].to_dict() for m in metrics_cols}

print('--- Best on cumulene_test.xyz ---')
for m, v in best_test.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")

print('--- Best on cumulene_profile.xyz ---')
for m, v in best_profile.items():
    print(f"{m}: {v['model']} ({v[m]:.6f})")


=== model eval on cumulene_test.xyz ===


,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A
1,CU-mp2,2.927394,0.556543,44.295026,13.697861
0,SOG-mp2,2.955400,0.935261,47.255143,17.848402
4,SOG-mp2-ldependent,3.052667,0.708406,46.926216,17.092449
5,SOG-mp1-ldependent,3.357204,1.093084,58.795308,27.021011
2,SOG-mp1,3.776757,1.521577,70.747145,34.675287
3,CU-mp1,3.859974,1.427199,67.607926,34.853146


=== model eval on cumulene_profile.xyz ===


,model,energy_rmse_meV_per_atom,energy_mae_meV_per_atom,force_rmse_meV_per_A,force_mae_meV_per_A
4,SOG-mp2-ldependent,0.240247,0.218822,18.937282,7.307531
0,SOG-mp2,0.431767,0.273755,13.809877,6.211781
1,CU-mp2,0.447121,0.178150,38.817495,6.018299
5,SOG-mp1-ldependent,0.741744,0.660449,30.413152,12.465875
2,SOG-mp1,0.950308,0.684109,34.254861,14.752613
3,CU-mp1,1.486348,1.432459,43.592531,17.484093


--- Best on cumulene_test.xyz ---
energy_mae_meV_per_atom: CU-mp2 (0.556543)
energy_rmse_meV_per_atom: CU-mp2 (2.927394)
force_mae_meV_per_A: CU-mp2 (13.697861)
force_rmse_meV_per_A: CU-mp2 (44.295026)
--- Best on cumulene_profile.xyz ---
energy_mae_meV_per_atom: CU-mp2 (0.178150)
energy_rmse_meV_per_atom: SOG-mp2-ldependent (0.240247)
force_mae_meV_per_A: CU-mp2 (6.018299)
force_rmse_meV_per_A: SOG-mp2 (13.809877)
